# Minimale ABox-Erweiterung fuer CQ-SPARQL-Queries

Dieses Notebook erzeugt eine kleine ABox-Erweiterung, die nur die Individuen enthaelt, die fuer positive Antworten der vorhandenen CQ-SPARQL-Queries benoetigt werden. Der Fokus liegt auf CQ12 `Insellappen` und CQ16 `Anastomose`.

Wichtig: CQ16 verwendet `rdfs:subClassOf+`. Deshalb wird kein Individuum direkt als `OFLID10223` typisiert, sondern als konkrete Technik-Subklasse, z.B. `OFLID10233` oder `OFLID10235`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import OWL, RDF, RDFS, XSD

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

OUT_TTL = ROOT / "ontologies" / "ofl_cq_minimal_abox_extension.ttl"
OUT_OWL = ROOT / "ontologies" / "ofl_cq_minimal_abox_extension.owl"
TBOX = ROOT / "ontologies" / "ofl_2.1.0_inferred.owl"

OFL = Namespace("https://purl.bioontology.org/ontology/OFL/")
OBO = Namespace("http://purl.obolibrary.org/obo/")
EX = Namespace("https://example.org/ofl/cq-abox/")

HAS_QUALITY = OBO.RO_0000086
HAS_MEASUREMENT_VALUE = OBO.IAO_0000004
HAS_MEASUREMENT_UNIT = OBO.IAO_0000039
IS_QUALITY_MEASURED_AS = OBO.OBI_0001938
VOLUME = OBO.PATO_0001679
MILLILITER = OBO.UO_0000095

## Minimale CQ-Antwortindividuen

Jeder Eintrag erzeugt genau ein Flap-Individuum mit einem direkten `rdf:type` auf der Antwortklasse, die die jeweilige Query prueft. So bleibt die ABox bewusst klein und vermeidet zusaetzliche Prozess- oder Anatomie-Individuen, solange die Query sie nicht benoetigt.

In [ ]:
CQ_FLAP_TYPES = [
    # CQ01 composition
    ("cq01_tissue_composition_flap", OFL.OFLID10102, "CQ01 minimal flap classified by tissue composition"),

    # CQ03 recipient vessel
    ("cq03_recipient_vessel_flap", OFL.OFLID130000, "CQ03 minimal flap classified by recipient vessel"),

    # CQ04 Mathes and Nahai
    ("cq04_mathes_nahai_flap", OFL.OFLID10135, "CQ04 minimal Mathes and Nahai muscle flap"),

    # CQ05 Nakajima-related buckets used in the query
    ("cq05_branch_recognized_perforator_flap", OFL.OFLID10075, "CQ05 branch-based flap with recognized perforator"),
    ("cq05_branch_based_flap", OFL.OFLID10076, "CQ05 branch-based flap"),
    ("cq05_perforator_based_flap", OFL.OFLID10078, "CQ05 perforator based flap"),

    # CQ06 distance
    ("cq06_local_flap", OFL.OFLID10134, "CQ06 local flap"),
    ("cq06_regional_flap", OFL.OFLID10140, "CQ06 regional flap"),
    ("cq06_distant_flap", OFL.OFLID10085, "CQ06 distant flap"),

    # CQ07 pedicled/free
    ("cq07_pedicled_flap", OFL.OFLID10053, "CQ07 pedicled flap"),
    ("cq07_free_flap", OFL.OFLID10054, "CQ07 free flap"),

    # CQ08 blood supply
    ("cq08_random_pattern_flap", OFL.OFLID10004, "CQ08 random pattern flap"),
    ("cq08_non_random_pattern_flap", OFL.OFLID10077, "CQ08 non-random pattern flap"),
    ("cq08_branch_based_flap", OFL.OFLID10076, "CQ08 branch-based flap"),
    ("cq08_branch_recognized_perforator_flap", OFL.OFLID10075, "CQ08 branch-based flap with recognized perforator"),
    ("cq08_perforator_based_flap", OFL.OFLID10078, "CQ08 perforator based flap"),

    # CQ09 chimeric
    ("cq09_chimeric_flap", OFL.OFLID1000092, "CQ09 chimeric flap"),

    # CQ10 flow direction
    ("cq10_anterograde_flow_flap", OFL.OFLID10113, "CQ10 flap with anterograde blood flow"),
    ("cq10_retrograde_flow_flap", OFL.OFLID10123, "CQ10 flap with retrograde blood flow"),

    # CQ11 movement
    ("cq11_rotation_flap", OFL.OFLID10014, "CQ11 rotation flap"),
    ("cq11_transposition_flap", OFL.OFLID10067, "CQ11 transposition flap"),
    ("cq11_advancement_flap", OFL.OFLID10070, "CQ11 advancement flap"),

    # CQ12 island flap - explicit focus of this notebook
    ("cq12_cutaneous_island_flap", OFL.OFLID10115, "CQ12 cutaneous island flap"),

    # CQ13 insertion site preparation
    ("cq13_insertion_site_preparation_flap", OFL.OFLID130001, "CQ13 flap classified by insertion site preparation"),

    # CQ14 pre-harvest modification
    ("cq14_without_preharvest_modification_flap", OFL.OFLID10130, "CQ14 flap without preharvest modification"),
    ("cq14_with_preharvest_modification_flap", OFL.OFLID10131, "CQ14 flap with preharvest modification"),

    # CQ15 skin graft
    ("cq15_skin_graft_flap", OFL.OFLID120066, "CQ15 flap with skin graft"),

    # CQ16 anastomosis technique - explicit focus of this notebook.
    # The CQ16 query requires a strict subclass of OFLID10223.
    ("cq16_sutured_anastomosis_flap", OFL.OFLID10233, "CQ16 flap with sutured vessel anastomosis"),
    ("cq16_coupled_anastomosis_flap", OFL.OFLID10235, "CQ16 flap with coupled vessel anastomosis"),

    # CQ17 survival
    ("cq17_without_tissue_loss_flap", OFL.OFLID10184, "CQ17 flap without tissue loss"),
    ("cq17_loss_at_apex_flap", OFL.OFLID10179, "CQ17 flap with loss at the apex"),
    ("cq17_total_loss_flap", OFL.OFLID10183, "CQ17 flap with total loss"),
    ("cq17_arterial_obstruction_flap", OFL.OFLID10180, "CQ17 flap with arterial obstruction"),
    ("cq17_venous_obstruction_flap", OFL.OFLID10181, "CQ17 flap with venous obstruction"),
]

pd.DataFrame(
    [(local, str(cls), label) for local, cls, label in CQ_FLAP_TYPES],
    columns=["individual", "direct_type", "label"],
)

## ABox schreiben

CQ2 braucht neben einem Flap genau eine Groessenqualitaet und eine Messung. Alle anderen CQ-Queries in `sparql/cq_questions_answers.sparql` koennen mit direkter Typisierung eines Flap-Individuums beantwortet werden.

In [ ]:
def bind_prefixes(graph: Graph) -> None:
    graph.bind("ofl", OFL)
    graph.bind("obo", OBO)
    graph.bind("ex", EX)
    graph.bind("owl", OWL)
    graph.bind("rdf", RDF)
    graph.bind("rdfs", RDFS)
    graph.bind("xsd", XSD)


def add_named_individual(graph: Graph, iri: URIRef, class_iri: URIRef, label: str) -> None:
    graph.add((iri, RDF.type, OWL.NamedIndividual))
    graph.add((iri, RDF.type, class_iri))
    graph.add((iri, RDFS.label, Literal(label, lang="en")))


abox = Graph()
bind_prefixes(abox)

for local_name, class_iri, label in CQ_FLAP_TYPES:
    add_named_individual(abox, EX[local_name], class_iri, label)

# CQ02: minimal concrete size answer.
size_flap = EX.cq02_sized_flap
size_quality = EX.cq02_volume_quality
size_measurement = EX.cq02_volume_measurement

add_named_individual(abox, size_flap, OFL.OFLID10002, "CQ02 flap with volume measurement")
add_named_individual(abox, size_quality, VOLUME, "CQ02 volume quality")
add_named_individual(abox, size_measurement, OBO.IAO_0000109, "CQ02 volume measurement datum")

abox.add((size_flap, HAS_QUALITY, size_quality))
abox.add((size_measurement, IS_QUALITY_MEASURED_AS, size_quality))
abox.add((size_measurement, HAS_MEASUREMENT_VALUE, Literal("42.0", datatype=XSD.decimal)))
abox.add((size_measurement, HAS_MEASUREMENT_UNIT, MILLILITER))

OUT_TTL.parent.mkdir(parents=True, exist_ok=True)
abox.serialize(destination=OUT_TTL, format="turtle")
abox.serialize(destination=OUT_OWL, format="xml")

print(f"Wrote {len(abox)} triples")
print(OUT_TTL.relative_to(ROOT))
print(OUT_OWL.relative_to(ROOT))

## Validierung der Fokus-CQs

Die Validierung laedt TBox plus ABox-Erweiterung. Dadurch kommen die `rdfs:subClassOf`-Ketten und Klassenlabels aus der TBox, waehrend die Antwortindividuen aus der minimalen ABox stammen.

In [ ]:
PREFIXES = """
PREFIX : <https://purl.bioontology.org/ontology/OFL/>
PREFIX obo: <http://purl.obolibrary.org/obo/>
PREFIX owl: <http://www.w3.org/2002/07/owl#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
"""

def query_df(graph: Graph, body: str) -> pd.DataFrame:
    result = graph.query(PREFIXES + body)
    cols = [str(v) for v in result.vars]
    return pd.DataFrame([{col: str(row[i]) for i, col in enumerate(cols)} for row in result])


validation_graph = Graph()
validation_graph.parse(TBOX)
validation_graph.parse(OUT_TTL, format="turtle")
print(f"Validation graph: {len(validation_graph)} triples")

In [ ]:
cq12 = """
SELECT DISTINCT ?flap ?flapLabel ?islandClass ?islandLabel
WHERE {
  VALUES ?islandClass { :OFLID10115 }
  ?flap rdf:type ?directType .
  ?directType rdfs:subClassOf* ?islandClass .
  FILTER(STRSTARTS(STR(?flap), "https://example.org/ofl/cq-abox/"))
  OPTIONAL { ?flap rdfs:label ?flapLabel . }
  ?islandClass rdfs:label ?islandLabel .
}
ORDER BY ?flapLabel
"""

display(query_df(validation_graph, cq12))

In [ ]:
cq16 = """
SELECT DISTINCT ?flap ?flapLabel ?anastomosisClass ?anastomosisLabel
WHERE {
  VALUES ?anastomosisClass { :OFLID10223 }
  ?flap rdf:type ?directType .
  ?directType rdfs:subClassOf+ ?anastomosisClass .
  FILTER(STRSTARTS(STR(?flap), "https://example.org/ofl/cq-abox/"))
  OPTIONAL { ?flap rdfs:label ?flapLabel . }
  ?anastomosisClass rdfs:label ?anastomosisLabel .
}
ORDER BY ?flapLabel
"""

display(query_df(validation_graph, cq16))

In [ ]:
cq16_detail = """
SELECT DISTINCT ?flap ?flapLabel ?techniqueClass ?techniqueLabel ?bucketLabel
WHERE {
  VALUES ?bucket { :OFLID10223 }
  ?flap rdf:type ?techniqueClass .
  ?techniqueClass rdfs:subClassOf+ ?bucket .
  FILTER(STRSTARTS(STR(?flap), "https://example.org/ofl/cq-abox/"))
  OPTIONAL { ?flap rdfs:label ?flapLabel . }
  OPTIONAL { ?techniqueClass rdfs:label ?techniqueLabel . }
  ?bucket rdfs:label ?bucketLabel .
}
ORDER BY ?flapLabel ?techniqueLabel
"""

display(query_df(validation_graph, cq16_detail))

## Optional: alle CQ-Queries gegen die Erweiterung testen

Diese Zelle splittet die bestehende Datei grob an den CQ-Kommentarbloecken und zaehlt, ob jede Query mindestens eine Antwort aus der `https://example.org/ofl/cq-abox/`-ABox liefert. Fuer eine strengere Pruefung koennen die einzelnen Query-Resultate angezeigt werden.

In [ ]:
sparql_file = ROOT / "sparql" / "cq_questions_answers.sparql"
text = sparql_file.read_text()

queries = []
for block in text.split("################################################################################"):
    if "SELECT" not in block or "WHERE" not in block:
        continue
    title = next((line.strip("# ") for line in block.splitlines() if line.strip().startswith("# CQ")), "CQ")
    query = block[block.index("SELECT"):].strip()
    queries.append((title, query))

rows = []
for title, query in queries:
    try:
        df = query_df(validation_graph, query)
        abox_rows = df.apply(lambda row: row.astype(str).str.contains("https://example.org/ofl/cq-abox/", regex=False).any(), axis=1) if not df.empty else []
        rows.append({"cq": title, "rows_total": len(df), "rows_from_minimal_abox": int(sum(abox_rows))})
    except Exception as exc:
        rows.append({"cq": title, "rows_total": None, "rows_from_minimal_abox": None, "error": str(exc)})

display(pd.DataFrame(rows))